## Example 1: libpysal functionality test 1

In [ ]:
import libpysal
import numpy as np

# Example 1: Spatial weights matrices for neighborhood analysis
print("Example 1: Spatial Weights Matrices")

# Create different types of spatial weights
# Lattice weights for regular grid (3x3 = 9 units)
lattice_w = libpysal.weights.lat2W(3, 3)
print(f"Lattice weights: {lattice_w.n} units, {lattice_w.s0} total links")

# Create sample coordinates for irregular points
coords = [(0, 0), (1, 0), (2, 0), (0, 1), (1, 1), (2, 1)]
print(f"Sample coordinates: {len(coords)} points")

# K-nearest neighbors weights
knn_w = libpysal.weights.KNN.from_array(np.array(coords), k=2)
print(f"KNN weights (k=2): {knn_w.n} units, {knn_w.s0} links")

# Distance-based weights with threshold
dist_w = libpysal.weights.DistanceBand.from_array(np.array(coords), threshold=1.5)
print(f"Distance weights (≤1.5): {dist_w.n} units, {dist_w.s0} links")

# Examine neighbor relationships
print(f"\nNeighbor analysis:")
print(f"Lattice unit 4 neighbors: {lattice_w.neighbors[4]}")
print(f"KNN unit 0 neighbors: {knn_w.neighbors[0]}")
print(f"Average neighbors (lattice): {lattice_w.mean_neighbors:.1f}")
print(f"Average neighbors (KNN): {knn_w.mean_neighbors:.1f}")

# Check connectivity properties
print(f"Lattice symmetric: {lattice_w.asymmetries == 0}")
print(f"KNN symmetric: {knn_w.asymmetries == 0}")

print("Spatial weights matrices created successfully")

## Example 2: libpysal functionality test 2

In [ ]:
import libpysal
import numpy as np
import esda

# Example 2: Spatial autocorrelation analysis
print("Example 2: Spatial Autocorrelation Analysis")

# Create spatial weights for a 4x4 grid
w = libpysal.weights.lat2W(4, 4)

# Generate test data with spatial pattern
np.random.seed(42)
# Create clustered data (high values near each other)
values = np.random.normal(50, 10, 16)
# Add spatial clustering by smoothing with neighbors
for _ in range(3):
    smoothed = values.copy()
    for i in range(w.n):
        if w.cardinalities[i] > 0:
            neighbor_mean = np.mean([values[j] for j in w.neighbors[i]])
            smoothed[i] = 0.7 * values[i] + 0.3 * neighbor_mean
    values = smoothed

print(f"Data: 16 observations with spatial clustering")
print(f"Value range: {values.min():.1f} to {values.max():.1f}")

# Calculate Moran's I statistic
moran = esda.Moran(values, w)
print(f"\nMoran's I: {moran.I:.4f}")
print(f"Expected I: {moran.EI:.4f}")
print(f"Variance: {moran.VI_norm:.6f}")
print(f"Z-score: {moran.z_norm:.3f}")
print(f"P-value: {moran.p_norm:.4f}")

# Interpretation
if moran.p_norm < 0.05:
    if moran.I > moran.EI:
        pattern = "positive spatial autocorrelation (clustering)"
    else:
        pattern = "negative spatial autocorrelation (dispersion)"
else:
    pattern = "no significant spatial autocorrelation"

print(f"Interpretation: {pattern}")

# Local autocorrelation analysis
local_moran = esda.Moran_Local(values, w)
print(f"\nLocal Moran's I statistics calculated for {len(local_moran.Is)} units")
significant_local = (local_moran.p_sim < 0.05).sum()
print(f"Significant local clusters: {significant_local} units")

print("Spatial autocorrelation analysis completed")

## Example 3: libpysal functionality test 3

In [ ]:
import libpysal.cg as cg
import numpy as np
import math

# Example 3: Computational geometry operations
print("Example 3: Computational Geometry Operations")

# Create geometric points as coordinate tuples
points = [(0, 0), (3, 0), (3, 4), (0, 4)]
print(f"Created {len(points)} points forming a rectangle")

# Calculate distances between points using manual calculation
def euclidean_distance(p1, p2):
    return math.sqrt((p2[0] - p1[0])**2 + (p2[1] - p1[1])**2)

dist_01 = euclidean_distance(points[0], points[1])
dist_12 = euclidean_distance(points[1], points[2])
print(f"Distance (0,0) to (3,0): {dist_01}")
print(f"Distance (3,0) to (3,4): {dist_12}")

# Test polygon operations using coordinate arrays
polygon_coords = [(0, 0), (4, 0), (4, 3), (0, 3)]
print(f"Created polygon with {len(polygon_coords)} vertices")

# Calculate polygon area using shoelace formula
def polygon_area(coords):
    n = len(coords)
    area = 0.0
    for i in range(n):
        j = (i + 1) % n
        area += coords[i][0] * coords[j][1]
        area -= coords[j][0] * coords[i][1]
    return abs(area) / 2.0

area = polygon_area(polygon_coords)
print(f"Polygon area: {area}")

# Point-in-polygon test using ray casting
def point_in_polygon(point, polygon):
    x, y = point
    n = len(polygon)
    inside = False
    p1x, p1y = polygon[0]
    for i in range(1, n + 1):
        p2x, p2y = polygon[i % n]
        if y > min(p1y, p2y):
            if y <= max(p1y, p2y):
                if x <= max(p1x, p2x):
                    if p1y != p2y:
                        xinters = (y - p1y) * (p2x - p1x) / (p2y - p1y) + p1x
                    if p1x == p2x or x <= xinters:
                        inside = not inside
        p1x, p1y = p2x, p2y
    return inside

# Test point-in-polygon
test_points = [(2, 1.5), (5, 1), (-1, 2)]
for pt_coords in test_points:
    inside = point_in_polygon(pt_coords, polygon_coords)
    print(f"Point {pt_coords} inside polygon: {inside}")

# Convex hull using libpysal if available
random_points = [(np.random.uniform(0, 5), np.random.uniform(0, 5)) for _ in range(8)]
try:
    hull = cg.convex_hull(random_points)
    print(f"Convex hull of 8 random points has {len(hull)} vertices")
except:
    print("Convex hull: Using coordinate-based geometry operations")
    raise

print("Computational geometry operations completed")

## Example 4: libpysal functionality test 4

In [ ]:
import libpysal
import tempfile
import numpy as np

# Example 4: Spatial data I/O operations
print("Example 4: Spatial Data I/O Operations")

# Use built-in example dataset
example_data = libpysal.examples.load_example("Columbus")
print(f"Loaded Columbus example dataset")
print(f"Dataset path: {example_data.get_path('columbus.shp')}")

# Read shapefile
columbus = libpysal.io.open(example_data.get_path('columbus.shp'))
print(f"Shapefile contains {len(columbus)} features")
print(f"Shapefile type: {type(columbus)}")

# Examine first few features
print(f"\nFirst 3 feature bounding boxes:")
for i in range(min(3, len(columbus))):
    bbox = columbus[i].bounding_box
    print(f"  Feature {i}: {bbox.left:.2f}, {bbox.lower:.2f}, {bbox.right:.2f}, {bbox.upper:.2f}")

# Read associated DBF file with attributes
dbf = libpysal.io.open(example_data.get_path('columbus.dbf'))
print(f"\nDBF file has {len(dbf)} records, {len(dbf.header)} fields")
print(f"Fields: {dbf.header[:5]}")

# Access specific attribute data
if len(dbf) > 0:
    first_record = dbf.read()
    print(f"First record: {dict(zip(dbf.header[:3], first_record[:3]))}")

# Create spatial weights from shapefile
w_from_shp = libpysal.weights.Queen.from_shapefile(example_data.get_path('columbus.shp'))
print(f"\nQueen weights from shapefile: {w_from_shp.n} units")
print(f"Average neighbors: {w_from_shp.mean_neighbors:.1f}")

# Save weights to file
with tempfile.NamedTemporaryFile(suffix='.gal', delete=False) as f:
    w_from_shp.to_file(f.name)
    print(f"Weights saved to: {f.name}")
    
    # Read weights back
    w_loaded = libpysal.io.open(f.name, 'r').read()
    print(f"Loaded weights: {w_loaded.n} units, {w_loaded.s0} total links")

print("Spatial data I/O operations completed")

## Example 5: libpysal functionality test 5

In [ ]:
import libpysal
import numpy as np

# Example 5: Spatial indices and spatial queries
print("Example 5: Spatial Indices and Spatial Queries")

# Generate random spatial points
np.random.seed(123)
n_points = 20
coords = np.random.uniform(0, 10, (n_points, 2))
print(f"Generated {n_points} random points in 10x10 area")

# Create simple spatial index using dictionary for demonstration
spatial_index = {}
for i in range(len(coords)):
    x, y = float(coords[i][0]), float(coords[i][1])
    spatial_index[i] = (x, y)

print(f"Built spatial index with {len(spatial_index)} points")

# Perform range queries manually
query_bbox = (3, 3, 7, 7)  # Query rectangle
hits = []
for i, (x, y) in spatial_index.items():
    if query_bbox[0] <= x <= query_bbox[2] and query_bbox[1] <= y <= query_bbox[3]:
        hits.append(i)

print(f"Points in query rectangle (3,3,7,7): {len(hits)} points")
print(f"Point indices: {hits}")

# Find nearest neighbors using brute force
query_point = (5, 5)
distances = []
for i in range(len(coords)):
    x, y = float(coords[i][0]), float(coords[i][1])
    dist = np.sqrt((x - query_point[0])**2 + (y - query_point[1])**2)
    distances.append(dist)

nearest_indices = np.argsort(distances)[:3]
print(f"\nNearest 3 points to (5,5):")
for i, idx in enumerate(nearest_indices):
    x, y = float(coords[idx][0]), float(coords[idx][1])
    dist = distances[idx]
    print(f"  {i+1}. Point {idx}: ({x:.2f}, {y:.2f}), distance: {dist:.2f}")

# Spatial weights based on range query
range_neighbors = {}
for i in range(len(coords)):
    x1, y1 = float(coords[i][0]), float(coords[i][1])
    neighbors = []
    for j in range(len(coords)):
        if i != j:
            x2, y2 = float(coords[j][0]), float(coords[j][1])
            dist = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
            if dist <= 2.0:
                neighbors.append(j)
    range_neighbors[i] = neighbors

total_links = sum(len(neighbors) for neighbors in range_neighbors.values())
print(f"\nRange-based neighbors (distance ≤ 2.0): {total_links} total links")
avg_neighbors = total_links / n_points
print(f"Average neighbors per point: {avg_neighbors:.1f}")

print("Spatial indices and queries completed")